# 01 — Análise Exploratória (EDA)
**Objetivo:** entender a distribuição dos dados, identificar valores faltantes, correlações e outliers antes de qualquer modelagem.

In [ ]:
import sys
sys.path.append('..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.dpi'] = 100
%matplotlib inline

## 1. Carregamento e visão geral

In [ ]:
train = pd.read_csv('../data/treino.csv')
test  = pd.read_csv('../data/teste_publico.csv')

print(f'Treino : {train.shape}  |  Teste: {test.shape}')
print(f'Colunas treino: {list(train.columns)}')

In [ ]:
# Tipos de dados e contagem de nulos
info_df = pd.DataFrame({
    'dtype': train.dtypes,
    'n_null': train.isnull().sum(),
    'pct_null': (train.isnull().mean() * 100).round(2)
}).sort_values('n_null', ascending=False)
print(info_df[info_df.n_null > 0])

In [ ]:
# Estatísticas descritivas das colunas numéricas
train.describe().T.style.background_gradient(cmap='Blues', subset=['mean','std','50%'])

## 2. Variável alvo — SalePrice

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 9))

# Histograma original
axes[0, 0].hist(train['SalePrice'], bins=50, color='steelblue', edgecolor='white')
axes[0, 0].set_title(f'SalePrice  |  skew={train["SalePrice"].skew():.2f}')
axes[0, 0].set_xlabel('USD')

# QQ-plot original
stats.probplot(train['SalePrice'], plot=axes[0, 1])
axes[0, 1].set_title('QQ-plot SalePrice')

# Histograma log1p
log_price = np.log1p(train['SalePrice'])
axes[1, 0].hist(log_price, bins=50, color='seagreen', edgecolor='white')
axes[1, 0].set_title(f'log1p(SalePrice)  |  skew={log_price.skew():.2f}')
axes[1, 0].set_xlabel('log(USD + 1)')

# QQ-plot log1p
stats.probplot(log_price, plot=axes[1, 1])
axes[1, 1].set_title('QQ-plot log1p(SalePrice)')

plt.tight_layout()
plt.show()

print(f'Skew original : {train["SalePrice"].skew():.4f}')
print(f'Skew log1p    : {log_price.skew():.4f}')
print(f'Kurtosis orig : {train["SalePrice"].kurtosis():.4f}')
print(f'Kurtosis log1p: {log_price.kurtosis():.4f}')

In [ ]:
# Boxplot para detectar outliers extremos
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
train[['SalePrice']].boxplot(ax=axes[0])
axes[0].set_title('Boxplot SalePrice')
np.log1p(train[['SalePrice']]).boxplot(ax=axes[1])
axes[1].set_title('Boxplot log1p(SalePrice)')
plt.tight_layout()
plt.show()

## 3. Valores Faltantes

In [ ]:
missing = train.isnull().mean().sort_values(ascending=False)
missing = missing[missing > 0] * 100

# Colunas onde NaN = "não tem" (definido pelo dicionário)
nan_means_none = [
    'Alley', 'PoolQC', 'Fence', 'MiscFeature', 'FireplaceQu',
    'GarageType', 'GarageFinish', 'GarageQual', 'GarageCond',
    'BsmtQual', 'BsmtCond', 'BsmtExposure', 'BsmtFinType1', 'BsmtFinType2',
    'MasVnrType',
]

classification = []
for col in missing.index:
    if col in nan_means_none:
        classification.append('NaN = não tem')
    else:
        classification.append('NaN = dado faltante')

missing_df = pd.DataFrame({
    'coluna': missing.index,
    '% missing': missing.values.round(1),
    'tipo': classification
})
print(missing_df.to_string(index=False))

In [ ]:
# Gráfico de barras dos faltantes
fig, ax = plt.subplots(figsize=(10, 6))
colors = ['coral' if t == 'NaN = não tem' else 'steelblue' for t in missing_df['tipo']]
ax.barh(missing_df['coluna'], missing_df['% missing'], color=colors)
ax.set_xlabel('% de valores faltantes')
ax.set_title('Valores Faltantes por Coluna\n(coral = NaN significa ausência; azul = dado realmente faltante)')
ax.invert_yaxis()
plt.tight_layout()
plt.show()

## 4. Correlações — Top 15 numéricas com SalePrice

In [ ]:
num_cols = train.select_dtypes(include=[np.number]).columns.tolist()
corr_matrix = train[num_cols].corr()
top15 = corr_matrix['SalePrice'].abs().sort_values(ascending=False).head(16).index.tolist()

fig, ax = plt.subplots(figsize=(12, 10))
mask = np.triu(np.ones_like(corr_matrix.loc[top15, top15], dtype=bool))
sns.heatmap(
    corr_matrix.loc[top15, top15],
    mask=mask,
    annot=True, fmt='.2f', cmap='coolwarm',
    center=0, linewidths=0.5, ax=ax
)
ax.set_title('Correlação — Top 15 features com SalePrice')
plt.tight_layout()
plt.show()

print('\nCorrelações com SalePrice (top 15):')
print(corr_matrix['SalePrice'].sort_values(ascending=False).head(16))

## 5. Scatter Plots — Top correlatas vs SalePrice

In [ ]:
top_feats = ['GrLivArea', 'TotalBsmtSF', 'OverallQual', 'GarageArea', '1stFlrSF']

fig, axes = plt.subplots(2, 3, figsize=(16, 9))
axes = axes.flatten()

for i, feat in enumerate(top_feats):
    if feat not in train.columns:
        continue
    # Outliers de GrLivArea: área grande mas preço baixo
    if feat == 'GrLivArea':
        outlier_mask = (train['GrLivArea'] > 4000) & (train['SalePrice'] < 300_000)
        axes[i].scatter(
            train.loc[~outlier_mask, feat], train.loc[~outlier_mask, 'SalePrice'],
            alpha=0.4, s=15, color='steelblue', label='Normal'
        )
        axes[i].scatter(
            train.loc[outlier_mask, feat], train.loc[outlier_mask, 'SalePrice'],
            alpha=0.9, s=60, color='crimson', marker='X', label='Outlier'
        )
        axes[i].legend()
    else:
        axes[i].scatter(train[feat], train['SalePrice'], alpha=0.3, s=15, color='steelblue')
    axes[i].set_xlabel(feat)
    axes[i].set_ylabel('SalePrice')
    r = train[[feat, 'SalePrice']].dropna().corr().iloc[0, 1]
    axes[i].set_title(f'{feat}  (r={r:.2f})')

axes[-1].axis('off')
plt.suptitle('Scatter: Features vs SalePrice', fontsize=13, y=1.01)
plt.tight_layout()
plt.show()

# Confirma os outliers
outliers_grlivarea = train[(train['GrLivArea'] > 4000) & (train['SalePrice'] < 300_000)]
print(f'Outliers GrLivArea > 4000 & SalePrice < 300k: {len(outliers_grlivarea)} linhas')
print(outliers_grlivarea[['GrLivArea', 'SalePrice', 'Neighborhood', 'OverallQual']])

## 6. Boxplots — SalePrice por Neighborhood e OverallQual

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(14, 12))

# Ordena bairros pela mediana do preço
order_neighborhood = (
    train.groupby('Neighborhood')['SalePrice']
    .median().sort_values(ascending=False).index
)
sns.boxplot(
    data=train, x='Neighborhood', y='SalePrice',
    order=order_neighborhood, ax=axes[0], palette='viridis'
)
axes[0].set_xticklabels(axes[0].get_xticklabels(), rotation=45, ha='right')
axes[0].set_title('SalePrice por Neighborhood (ordenado por mediana)')

# OverallQual
sns.boxplot(
    data=train, x='OverallQual', y='SalePrice',
    ax=axes[1], palette='RdYlGn'
)
axes[1].set_title('SalePrice por OverallQual')

plt.tight_layout()
plt.show()

## 7. Cardinalidade das Categóricas

In [ ]:
cat_cols = train.select_dtypes(include='object').columns.tolist()

card_df = pd.DataFrame({
    'coluna': cat_cols,
    'n_categorias': [train[c].nunique() for c in cat_cols],
    'classe_dominante_pct': [
        train[c].value_counts(normalize=True).iloc[0] * 100 for c in cat_cols
    ],
    'classe_dominante': [
        train[c].value_counts().index[0] for c in cat_cols
    ]
}).sort_values('n_categorias', ascending=False)

print('Cardinalidade das colunas categóricas:')
print(card_df.to_string(index=False))
print(f'\nColunas com classe dominante > 95%:')
print(card_df[card_df['classe_dominante_pct'] > 95][['coluna', 'classe_dominante', 'classe_dominante_pct']].to_string(index=False))

## 8. Estratégia de Limpeza

### Outliers a remover
- **2 observações** com `GrLivArea > 4000` e `SalePrice < 300k` — imóveis grandes mas baratos, provavelmente venda em condições especiais. Removidos antes do treino.

### Variável alvo
- Aplicar `log1p(SalePrice)` no treino para normalizar a distribuição (skew ~1.88 → ~0.12). A métrica oficial é RMSLE, que equivale a RMSE no espaço log.

### Imputação por tipo de coluna
| Tipo | Estratégia |
|------|------------|
| NaN = "não tem" (Alley, PoolQC, Fence, FireplaceQu, Garage\*, Bsmt\*, MasVnrType) | `fillna('None')` antes do encoding |
| Numéricas com NaN real (LotFrontage, MasVnrArea, GarageYrBlt, etc.) | `SimpleImputer(strategy='median')` |
| Categóricas com NaN real (Electrical) | `SimpleImputer(strategy='most_frequent')` |

### Encoding planejado
| Tipo | Colunas | Encoding |
|------|---------|----------|
| Ordinal qualidade (Ex/Gd/TA/Fa/Po) | ExterQual, ExterCond, BsmtQual, BsmtCond, HeatingQC, KitchenQual, FireplaceQu, GarageQual, GarageCond, PoolQC | Mapeamento manual Ex=5…Po=1, None=0 |
| Nominal baixa cardinalidade | Alley, GarageType, Fence, MiscFeature, etc. | OneHotEncoder(handle_unknown='ignore') |
| Nominal alta cardinalidade (Neighborhood) | Neighborhood | OneHotEncoder |

### Escalonamento
- `StandardScaler` apenas nas colunas numéricas (após imputação).

### Features novas (`src/feature_engineering.py`)
| Feature | Descrição |
|---------|----------|
| `TotalSF` | TotalBsmtSF + 1stFlrSF + 2ndFlrSF |
| `TotalBath` | Full + 0.5×Half + BsmtFull + 0.5×BsmtHalf |
| `HouseAge` | YrSold − YearBuilt |
| `RemodAge` | YrSold − YearRemodAdd |
| `HasPool` | PoolArea > 0 |
| `HasGarage` | GarageArea > 0 |
| `HasFireplace` | Fireplaces > 0 |
| `HasBsmt` | TotalBsmtSF > 0 |